# 교통안전 EPDO 통합 분석 — 2부B (STEP 09 ~ STEP 11)

엔트로피 가중치 / 회귀 가중치 / 시설물 설치 효과
**2부A 완료 후 실행하세요**

In [ ]:
import csv, json, math, os, warnings
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, roc_auc_score)
from sklearn.model_selection import train_test_split
import geopandas as gpd
from shapely.geometry import Point

try:
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except ImportError:
    from sklearn.linear_model import PoissonRegressor
    HAS_STATSMODELS = False

warnings.filterwarnings("ignore", category=UserWarning)

# ── 경로 설정 ──────────────────────────────────────────────────────────────────
# COMPAS 구조: 노트북과 같은 위치에 data/ 폴더가 있음
BASE_DIR   = os.getcwd()          # 노트북 실행 위치 (data/ 폴더의 부모)
EPDO_DIR   = os.getcwd()          # 분석 결과 저장 위치
OUTPUT_DIR = os.path.join(EPDO_DIR, "epdo_analysis", "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"BASE_DIR  : {BASE_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

---

## 09_ENTROPY_WEIGHT

# STEP 09 - 엔트로피 가중법으로 보정 계수 가중치 산출

- 입력: epdo_analysis/output/08_격자_종합위험지수.csv
- 출력: epdo_analysis/output/09_엔트로피_가중치.csv

엔트로피 가중법 (Entropy Weight Method):
  변동이 클수록 정보량이 많다 → 가중치 높음
  변동이 작을수록 정보량이 없다 → 가중치 낮음

4단계:
  1. 정규화: 각 인자를 0~1 범위로 통일
  2. 비율 계산: 각 격자의 인자가 전체에서 차지하는 비중
  3. 엔트로피 계산: 값이 균일할수록 엔트로피 → 1 (정보 없음)
  4. 가중치 산출: (1 - 엔트로피) 정규화

In [ ]:
import csv
import math
import os

INPUT_PATH  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "08_격자_종합위험지수.csv")
OUTPUT_PATH = os.path.join(EPDO_DIR, "epdo_analysis", "output", "09_엔트로피_가중치.csv")

# ── 가중치를 산출할 인자 목록 ─────────────────────────────────────────────────
# composite_risk 공식의 7개 보정 인자
FACTORS = {
    "elderly_res_ratio":  "노인거주비율     (03 거주인구)",
    "vuln_float_ratio":   "교통약자유동비율  (04 유동인구)",
    "gap_cnt":            "인프라공백수     (STEP 06)",
    "grid_avg_speed":     "격자평균속도     (09 속도)",
    "grid_congestion":    "혼잡위험도       (11·12 혼잡강도)",
    "vuln_peak_pop":      "취약시간대인구   (05·06 시간대인구)",
    "weekend_ratio":      "주말방문비율     (07 서비스인구)",
}

FACTOR_KEYS = list(FACTORS.keys())


# ── 통계 유틸 ─────────────────────────────────────────────────────────────────

def normalize_minmax(values):
    """최솟값-최댓값 정규화 → 0~1"""
    v_min = min(values)
    v_max = max(values)
    denom = v_max - v_min
    if denom == 0:
        return [0.5] * len(values)   # 모두 같으면 0.5로 처리
    return [(v - v_min) / denom for v in values]


def entropy_weight(matrix):
    """
    matrix: dict { factor_key: [값1, 값2, ..., 값n] }
    반환  : dict { factor_key: weight }
    """
    n = len(next(iter(matrix.values())))   # 격자 수
    k = 1.0 / math.log(n)                 # 정규화 상수

    entropies = {}
    for key, values in matrix.items():
        norm = normalize_minmax(values)

        # 비율 계산 (0 방지를 위해 tiny 값 추가)
        total = sum(norm) or 1.0
        p = [v / total for v in norm]

        # 엔트로피: ej = -k × Σ(pij × ln(pij))
        e = 0.0
        for pi in p:
            if pi > 0:
                e += pi * math.log(pi)
        entropies[key] = -k * e

    # 편차 dj = 1 - ej
    deviations = {k: 1.0 - e for k, e in entropies.items()}

    # 가중치 wj = dj / Σdj
    total_dev = sum(deviations.values())
    weights = {k: round(d / total_dev, 6) for k, d in deviations.items()}

    return entropies, deviations, weights


# ── 메인 ─────────────────────────────────────────────────────────────────────

print("=" * 60)
print("STEP 09 - 엔트로피 가중법 가중치 산출")
print("=" * 60)

# 1. 데이터 로드
print("\n[1] 데이터 로드 중...")
rows = []
with open(INPUT_PATH, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        rows.append(row)
print(f"    격자 수: {len(rows):,}개")

# 2. 인자별 값 추출 (빈 값은 0 또는 평균 대체)
print("\n[2] 인자별 데이터 추출 중...")
matrix = {}
missing_info = {}

for key in FACTOR_KEYS:
    raw = []
    missing = 0
    for row in rows:
        val = row.get(key, "")
        if val == "" or val is None:
            missing += 1
            raw.append(None)
        else:
            try:
                raw.append(float(val))
            except ValueError:
                missing += 1
                raw.append(None)

    # 결측값 처리: 비결측 평균으로 대체
    valid = [v for v in raw if v is not None]
    avg = sum(valid) / len(valid) if valid else 0.0
    filled = [v if v is not None else avg for v in raw]

    matrix[key] = filled
    missing_info[key] = missing

for key, miss in missing_info.items():
    pct = miss / len(rows) * 100
    fill_note = f"(결측 {miss}개 → 평균 {sum(v for v in matrix[key])/len(matrix[key]):.4f}으로 대체)" if miss > 0 else ""
    print(f"    {key:25s}: {len(rows)-miss:4d}개 유효  {fill_note}")

# 3. 엔트로피 가중치 계산
print("\n[3] 엔트로피 가중치 계산 중...")
entropies, deviations, weights = entropy_weight(matrix)

# 가중치 기준 내림차순 정렬
sorted_keys = sorted(weights, key=lambda k: -weights[k])

# 4. 결과 출력
print("\n" + "=" * 60)
print("  [결과] 인자별 엔트로피 가중치")
print("=" * 60)
print(f"\n  {'순위':>3}  {'인자':25s}  {'엔트로피':>10}  {'편차':>8}  {'가중치':>8}  설명")
print("  " + "-" * 80)

result = []
for rank, key in enumerate(sorted_keys, 1):
    desc = FACTORS[key]
    e    = entropies[key]
    d    = deviations[key]
    w    = weights[key]
    bar  = "█" * int(w * 100)

    print(f"  {rank:>3}. {key:25s}  {e:10.6f}  {d:8.6f}  {w:8.4%}  {desc}")
    print(f"       {bar}")

    result.append({
        "순위":   rank,
        "인자":   key,
        "설명":   desc,
        "엔트로피": round(e, 6),
        "편차":   round(d, 6),
        "가중치": round(w, 6),
        "가중치(%)": f"{w*100:.2f}%",
    })

print()

# 5. 현재 공식 vs 가중치 적용 공식 비교
print("=" * 60)
print("  [현재 공식] — 모든 인자 동등 가중 (곱셈)")
print("=" * 60)
print("""
  composite_risk = EPDO
× vuln_correction    (노인거주 + 교통약자유동)
× infra_penalty      (인프라공백)
× speed_weight       (속도)
× congestion_factor  (혼잡강도)
× peak_factor        (취약시간 인구)
× weekend_factor     (주말비율)
""")

print("=" * 60)
print("  [가중치 적용 공식] — 엔트로피 가중합")
print("=" * 60)

# 가중합 방식으로 변환
factor_map = {
    "elderly_res_ratio": "elderly_res_ratio",
    "vuln_float_ratio":  "vuln_float_ratio",
    "gap_cnt":           "gap_cnt / 9",
    "grid_avg_speed":    "grid_avg_speed / 60",
    "grid_congestion":   "grid_congestion / 100",
    "vuln_peak_pop":     "peak_factor - 1",
    "weekend_ratio":     "weekend_ratio",
}

print("\n  보정지수 = Σ(가중치 × 정규화된 인자값)")
print()
for key in sorted_keys:
    w = weights[key]
    expr = factor_map.get(key, key)
    print(f"    {w:.4%} × {expr:35s}  ({FACTORS[key]})")

print("""
  composite_risk = EPDO × (1 + 보정지수)

  ※ 1을 더하는 이유: EPDO가 0인 격자도 보정값이 0보다 크게 만들기 위함
""")

# 6. 해석 안내
print("=" * 60)
print("  [해석 가이드]")
print("=" * 60)
top3 = sorted_keys[:3]
low3 = sorted_keys[-3:]
print(f"\n  가중치 상위 3개 (가장 중요한 인자):")
for k in top3:
    print(f"    → {k}: {weights[k]:.4%}  ─  {FACTORS[k]}")
print(f"\n  가중치 하위 3개 (덜 중요한 인자):")
for k in low3:
    print(f"    → {k}: {weights[k]:.4%}  ─  {FACTORS[k]}")

print("""
  ※ 가중치가 높다 = 격자마다 값이 많이 달랐다 = 차별화 능력이 높다
  ※ 가중치가 낮다 = 격자마다 값이 비슷했다  = 차별화 능력이 낮다
  ※ 이는 "더 중요한 인자"가 아닌 "더 많이 변동하는 인자"임을 주의
 전문가 판단(AHP)과 병행하면 더 신뢰성 높은 가중치 산출 가능
""")

# 7. CSV 저장
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
cols = list(result[0].keys())
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result)

print(f"저장: {OUTPUT_PATH}")
print("=" * 60)

---

## 10_NB_REGRESSION_WEIGHT

# STEP 10 - 음이항 회귀 주축 + 엔트로피/상관계수 3중 교차검증 통합 가중치

목적:
  엔트로피 단독의 한계(변별력 ≠ 중요도)를 보완하기 위해
  음이항 회귀(NB), EPDO 상관계수(Corr), 엔트로피(Entropy) 3가지 방법으로
  각 인자의 가중치를 산출하고, 기하평균으로 통합한다.

각 방법의 역할:
  - NB 회귀    : "이 인자가 사고 건수에 실제로 얼마나 영향을 주나?" (주축)
  - Corr 가중  : "이 인자가 사고 심각도(EPDO)와 얼마나 연동되나?" (교차검증 1)
  - 엔트로피   : "이 인자가 격자를 얼마나 잘 구분하나?"           (교차검증 2)

통합 방식:
  w_final = geometric_mean(w_nb, w_corr, w_entropy) → 정규화

입력: epdo_analysis/output/08_격자_종합위험지수.csv
      epdo_analysis/output/09_엔트로피_가중치.csv
출력: epdo_analysis/output/10_통합가중치_최종.csv

In [ ]:
import csv
import math
import os

import numpy as np
import pandas as pd

# statsmodels 음이항 회귀 (없으면 Poisson fallback)
try:
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except ImportError:
    from sklearn.linear_model import PoissonRegressor
    from sklearn.preprocessing import StandardScaler
    HAS_STATSMODELS = False

INPUT_08      = os.path.join(EPDO_DIR, "epdo_analysis", "output", "08_격자_종합위험지수.csv")
INPUT_09      = os.path.join(EPDO_DIR, "epdo_analysis", "output", "09_엔트로피_가중치.csv")
OUTPUT_PATH   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "10_통합가중치_최종.csv")

FACTORS = {
    "elderly_res_ratio": "노인거주비율",
    "vuln_float_ratio":  "교통약자유동비율",
    "gap_cnt":           "인프라공백수",
    "grid_avg_speed":    "격자평균속도",
    "grid_congestion":   "혼잡위험도",
    "vuln_peak_pop":     "취약시간대인구",
    "weekend_ratio":     "주말방문비율",
}
FACTOR_KEYS = list(FACTORS.keys())

# grid_avg_speed 는 사고와 반대 방향 → 절댓값 처리 명시
NEGATIVE_EXPECTED = {"grid_avg_speed"}


# ── 유틸 ──────────────────────────────────────────────────────────────────────

def normalize(d: dict) -> dict:
    """dict 값을 합계=1로 정규화"""
    total = sum(d.values())
    return {k: v / total for k, v in d.items()}


def geometric_mean_weights(*weight_dicts):
    """
    여러 가중치 dict의 기하평균 → 정규화
    기하평균: 각 인자마다 모든 방법 가중치를 곱해 n제곱근 취함
    """
    n = len(weight_dicts)
    keys = list(weight_dicts[0].keys())
    raw = {}
    for k in keys:
        product = 1.0
        for wd in weight_dicts:
            product *= wd[k]
        raw[k] = product ** (1.0 / n)
    return normalize(raw)


def spearman_corr(rank_a, rank_b):
    """두 순위 리스트의 스피어만 상관계수"""
    n = len(rank_a)
    d2 = sum((a - b) ** 2 for a, b in zip(rank_a, rank_b))
    return 1 - (6 * d2) / (n * (n ** 2 - 1))


# ── 1단계: 데이터 로드 ─────────────────────────────────────────────────────────

def load_data():
    df = pd.read_csv(INPUT_08, encoding="utf-8-sig")
    print(f"    총 격자 수: {len(df):,}개")

    # 결측값 처리: 비결측 평균으로 대체
    cols_needed = FACTOR_KEYS + ["accident_cnt", "epdo_total"]
    for col in cols_needed:
        if col in df.columns:
            missing = df[col].isna().sum()
            if missing > 0:
                df[col] = df[col].fillna(df[col].mean())
                print(f"    {col}: 결측 {missing}개 → 평균 대체")
        else:
            print(f"    [경고] {col} 컬럼 없음 → 0으로 채움")
            df[col] = 0.0

    # accident_cnt는 정수형
    df["accident_cnt"] = df["accident_cnt"].clip(lower=0).round().astype(int)
    return df


# ── 2단계: 음이항 회귀 → NB 가중치 ───────────────────────────────────────────

def nb_regression_weights(df):
    X_raw = df[FACTOR_KEYS].values
    y     = df["accident_cnt"].values

    # 표준화 (StandardScaler): 계수가 변수 간 직접 비교 가능해짐
    mu    = X_raw.mean(axis=0)
    sigma = X_raw.std(axis=0)
    sigma[sigma == 0] = 1.0
    X_std = (X_raw - mu) / sigma

    if HAS_STATSMODELS:
        print("    [모델] statsmodels 음이항(NegativeBinomial) 회귀")
        X_sm = sm.add_constant(X_std)
        model = sm.NegativeBinomial(y, X_sm)
        result = model.fit(disp=False, maxiter=200)
        coefs = result.params[1:]          # 상수항 제외
        pvals = result.pvalues[1:]
        model_name = "음이항 회귀 (NegativeBinomial)"
    else:
        print("    [모델] sklearn PoissonRegressor (statsmodels 미설치)")
        from sklearn.linear_model import PoissonRegressor
        reg = PoissonRegressor(max_iter=500)
        reg.fit(X_std, y)
        coefs = reg.coef_
        pvals = [None] * len(coefs)
        model_name = "푸아송 회귀 (Poisson, NB 근사)"

    # 표준화 계수 절댓값 → 정규화
    abs_coefs = {k: abs(float(coefs[i])) for i, k in enumerate(FACTOR_KEYS)}
    weights   = normalize(abs_coefs)

    return weights, abs_coefs, pvals, model_name


# ── 3단계: EPDO 상관계수 → Corr 가중치 ───────────────────────────────────────

def corr_weights(df):
    corrs = {}
    for key in FACTOR_KEYS:
        c = df[key].corr(df["epdo_total"])
        corrs[key] = abs(c)

    weights = normalize(corrs)
    return weights, corrs


# ── 4단계: 엔트로피 가중치 로드 ───────────────────────────────────────────────

def load_entropy_weights():
    entropy_w = {}
    with open(INPUT_09, "r", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            key = row["인자"]
            if key in FACTOR_KEYS:
                entropy_w[key] = float(row["가중치"])
    # 혹시 누락된 인자 있으면 균등값으로 채움
    for k in FACTOR_KEYS:
        if k not in entropy_w:
            entropy_w[k] = 1.0 / len(FACTOR_KEYS)
    return normalize(entropy_w)


# ── 5단계: 합의도(★) 판정 ────────────────────────────────────────────────────

def consensus_star(nb_rank, corr_rank, ent_rank, n=7):
    """상위 절반(top 4/7) 기준 합의도"""
    threshold = n // 2 + 1   # 4
    top_count = sum([
        nb_rank   <= threshold,
        corr_rank <= threshold,
        ent_rank  <= threshold,
    ])
    return "★★★" if top_count == 3 else "★★" if top_count == 2 else "★"


# ── 메인 ─────────────────────────────────────────────────────────────────────

sep = "=" * 70
print(sep)
print("STEP 10 - 3중 교차검증 통합 가중치 산출")
print(sep)

# 1. 데이터 로드
print("\n[1] 데이터 로드")
df = load_data()

# 2. 음이항 회귀
print("\n[2] 음이항 회귀 가중치 계산")
nb_w, nb_coefs, pvals, model_name = nb_regression_weights(df)
print(f"    사용 모델: {model_name}")

# 3. 상관계수 가중치
print("\n[3] EPDO 상관계수 가중치 계산")
corr_w, corrs = corr_weights(df)
for k in FACTOR_KEYS:
    direction = "↑" if corrs[k] >= 0 else "↓"
    print(f"    {k:25s}: r={df[k].corr(df['epdo_total']):+.4f} {direction}  → |r|={corrs[k]:.4f}")

# 4. 엔트로피 가중치
print("\n[4] 엔트로피 가중치 로드")
ent_w = load_entropy_weights()
for k in FACTOR_KEYS:
    print(f"    {k:25s}: {ent_w[k]:.6f}")

# 5. 통합 가중치 (기하평균)
print("\n[5] 통합 가중치 계산 (기하평균)")
final_w = geometric_mean_weights(nb_w, corr_w, ent_w)

# 6. 순위 산출
nb_rank   = {k: sorted(nb_w,   key=lambda x: -nb_w[x]).index(k)  + 1 for k in FACTOR_KEYS}
corr_rank = {k: sorted(corr_w, key=lambda x: -corr_w[x]).index(k)+ 1 for k in FACTOR_KEYS}
ent_rank  = {k: sorted(ent_w,  key=lambda x: -ent_w[x]).index(k) + 1 for k in FACTOR_KEYS}
final_rank= {k: sorted(final_w,key=lambda x: -final_w[x]).index(k)+1 for k in FACTOR_KEYS}

# 7. 결과 출력
print("\n" + sep)
print("  [결과] 3중 교차검증 통합 가중치")
print(sep)
header = f"  {'인자':25s}  {'NB회귀':>8}  {'EPDO상관':>8}  {'엔트로피':>8}  {'통합':>8}  합의도  설명"
print(f"\n{header}")
print("  " + "-" * 85)

sorted_keys = sorted(FACTOR_KEYS, key=lambda k: -final_w[k])
result_rows = []

for k in sorted_keys:
    star = consensus_star(nb_rank[k], corr_rank[k], ent_rank[k])
    desc = FACTORS[k]
    pval_str = f"p={float(pvals[FACTOR_KEYS.index(k)]):.3f}" if pvals[0] is not None else ""
    print(
        f"  {k:25s}  {nb_w[k]:8.4%}  {corr_w[k]:8.4%}  {ent_w[k]:8.4%}"
        f"  {final_w[k]:8.4%}  {star:4s}  {desc}"
    )
    result_rows.append({
        "최종순위":     final_rank[k],
        "인자":         k,
        "설명":         desc,
        "NB회귀_가중치":   round(nb_w[k],  6),
        "NB회귀_순위":     nb_rank[k],
        "EPDO상관_가중치": round(corr_w[k], 6),
        "EPDO상관_순위":   corr_rank[k],
        "엔트로피_가중치": round(ent_w[k],  6),
        "엔트로피_순위":   ent_rank[k],
        "통합가중치":    round(final_w[k], 6),
        "합의도":        star,
    })

# 8. 스피어만 순위 상관 (방법 간 일치도)
print(f"\n{'─'*50}")
print("  [방법 간 스피어만 순위 상관계수]")
nb_r   = [nb_rank[k]   for k in FACTOR_KEYS]
corr_r = [corr_rank[k] for k in FACTOR_KEYS]
ent_r  = [ent_rank[k]  for k in FACTOR_KEYS]
sc_nb_corr = spearman_corr(nb_r, corr_r)
sc_nb_ent  = spearman_corr(nb_r, ent_r)
sc_corr_ent= spearman_corr(corr_r, ent_r)
print(f"  NB회귀 ↔ EPDO상관:  {sc_nb_corr:+.4f}")
print(f"  NB회귀 ↔ 엔트로피:  {sc_nb_ent:+.4f}")
print(f"  EPDO상관 ↔ 엔트로피: {sc_corr_ent:+.4f}")

# 9. 핵심 변수 요약
core    = [k for k in sorted_keys if consensus_star(nb_rank[k], corr_rank[k], ent_rank[k]) == "★★★"]
major   = [k for k in sorted_keys if consensus_star(nb_rank[k], corr_rank[k], ent_rank[k]) == "★★"]
ref     = [k for k in sorted_keys if consensus_star(nb_rank[k], corr_rank[k], ent_rank[k]) == "★"]

print(f"\n{'─'*50}")
print("  [합의도 기반 변수 분류]")
print(f"\n  ★★★ 핵심변수 (3가지 방법 모두 상위 4위 이내):")
for k in core:
    print(f"    → {k}: 통합 {final_w[k]:.2%}  ({FACTORS[k]})")
print(f"\n  ★★  주요변수 (2가지 방법 상위 4위 이내):")
for k in major:
    print(f"    → {k}: 통합 {final_w[k]:.2%}  ({FACTORS[k]})")
print(f"\n  ★   참고변수 (1가지 이하):")
for k in ref:
    print(f"    → {k}: 통합 {final_w[k]:.2%}  ({FACTORS[k]})")

# 10. 엔트로피 단독 대비 변화
print(f"\n{'─'*50}")
print("  [엔트로피 단독 → 통합 가중치 순위 변화]")
ent_sorted  = sorted(FACTOR_KEYS, key=lambda k: -ent_w[k])
final_sorted= sorted(FACTOR_KEYS, key=lambda k: -final_w[k])
for k in FACTOR_KEYS:
    old = ent_rank[k]
    new = final_rank[k]
    delta = old - new
    arrow = f"▲{delta}" if delta > 0 else f"▼{abs(delta)}" if delta < 0 else "─"
    print(f"  {k:25s}: 엔트로피 {old}위 → 통합 {new}위  {arrow}")

print(f"\n  통합 가중치 합계: {sum(final_w.values()):.6f}  (1.0 검증)")
print()

# 11. CSV 저장
result_rows.sort(key=lambda r: r["최종순위"])
cols = list(result_rows[0].keys())
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result_rows)

print(f"저장: {OUTPUT_PATH}")
print(sep)

---

## 11_FACILITY_EFFECT

# STEP 11: 스마트 안전시설물 설치 효과 계량화

=============================================
문헌 기반 시설물별 사고 감소율을 적용하여
고위험 격자별 시설물 설치 후 예상 EPDO 감소량을 산출한다.

입력:
  epdo_analysis/output/06_인프라공백_고위험격자.csv
  epdo_analysis/output/08_격자_종합위험지수.csv   (entropy_rank 참조)

출력:
  epdo_analysis/output/11_시설물설치_효과예측.csv

방법론:
  - 문헌 기반 보수적 감소율 적용 (실증 전후 비교 데이터 부재 시 표준 접근법)
  - 독립 효과 가정: 총 감소율 = 1 - Π(1 - r_i)
  - 효율성 = EPDO 감소량 / 공백 시설물 수 (시설물 1개당 기대 효과)
  - 설치 우선순위 = 효율성 기준 내림차순

출처 근거 (보수적 하한값 적용):
  - 횡단보도 15%: 교통안전공단 (보행자 사고 15~25% 감소)
  - 어린이보호구역 25%: 행정안전부 (어린이 사고 30~40% 감소)
  - 과속방지턱 20%: 도로교통공단 (사고 20~30% 감소)
  - CCTV 개소 12%: 경찰청 (무인 단속 효과 10~20%)
  - CCTV 대수 5%: CCTV 개소와 중복 효과 최소화
  - 버스정류장 8%: 보행자 대기 공간 정비 효과

In [ ]:
import csv
import os

# ── 경로 설정 ─────────────────────────────────────────────────────────────────
INPUT_GAP   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "06_인프라공백_고위험격자.csv")
INPUT_RISK  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "08_격자_종합위험지수.csv")
OUTPUT_PATH = os.path.join(EPDO_DIR, "epdo_analysis", "output", "11_시설물설치_효과예측.csv")

# ── 시설물별 문헌 기반 보수적 사고 감소율 ──────────────────────────────────────
REDUCTION_RATES = {
    "crosswalk_cnt":  0.15,   # 횡단보도: 교통안전공단 15~25% → 하한 15%
    "child_zone_cnt": 0.25,   # 어린이보호구역: 행안부 30~40% → 하한 25%
    "speedbump_cnt":  0.20,   # 과속방지턱: 도로교통공단 20~30% → 하한 20%
    "cctv_cnt":       0.12,   # CCTV 개소: 경찰청 10~20% → 하한 12%
    "cctv_cam_cnt":   0.05,   # CCTV 대수: 개소 효과와 중복 최소화 → 5%
    "bus_stop_cnt":   0.08,   # 버스정류장: 보행자 대기 공간 정비 → 8%
}

FACILITY_KR = {
    "crosswalk_cnt":  "횡단보도",
    "child_zone_cnt": "어린이보호구역",
    "speedbump_cnt":  "과속방지턱",
    "cctv_cnt":       "CCTV(개소)",
    "cctv_cam_cnt":   "CCTV(대수)",
    "bus_stop_cnt":   "버스정류장",
}


def parse_gap_items(gap_str):
    """gap_items 문자열에서 공백 시설물 목록을 추출.
    예: "crosswalk_cnt(없음) | cctv_cnt(없음)" → {'crosswalk_cnt', 'cctv_cnt'}
    """
    if not gap_str or gap_str.strip() == "":
        return set()
    missing = set()
    for item in gap_str.split("|"):
        item = item.strip()
        if "(" in item:
            col = item.split("(")[0].strip()
            if col in REDUCTION_RATES:
                missing.add(col)
    return missing


def combined_reduction(missing_facilities):
    """독립 효과 가정: 총 감소율 = 1 - Π(1 - r_i)"""
    survive = 1.0
    for fac in missing_facilities:
        r = REDUCTION_RATES.get(fac, 0.0)
        survive *= (1.0 - r)
    return round(1.0 - survive, 6)

## ── 1. 08번 파일에서 entropy_rank 로드

In [ ]:
print("[1] entropy_rank 로드 중...")
entropy_rank_map = {}
entropy_risk_map = {}
with open(INPUT_RISK, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        gid = row["grid_gid"]
        entropy_rank_map[gid] = row.get("entropy_rank", "")
        entropy_risk_map[gid] = row.get("entropy_composite_risk", "")
print(f"    로드 완료: {len(entropy_rank_map):,}개 격자")

## ── 2. 06번 공백 파일 로드 및 효과 계산

In [ ]:
print("\n[2] 시설물 설치 효과 계산 중...")
results = []
facility_epdo_saved = {fac: 0.0 for fac in REDUCTION_RATES}

with open(INPUT_GAP, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    for row in reader:
        gid       = row["grid_gid"]
        epdo_before = float(row.get("epdo_total", 0) or 0)
        acc_cnt   = int(row.get("accident_cnt", 0) or 0)
        death_cnt = int(row.get("사망_cnt", 0) or 0)
        injury_cnt= int(row.get("중상_cnt", 0) or 0)
        gap_cnt   = int(row.get("gap_cnt", 0) or 0)
        gap_items = row.get("gap_items", "")

        # 공백 시설물 파싱
        missing = parse_gap_items(gap_items)

        # 총 감소율 산출
        total_reduction = combined_reduction(missing)

        # 설치 후 예상 EPDO
        epdo_after = round(epdo_before * (1.0 - total_reduction), 2)
        epdo_saved = round(epdo_before - epdo_after, 2)

        # 효율성: 시설물 1개당 EPDO 감소량
        efficiency = round(epdo_saved / gap_cnt, 4) if gap_cnt > 0 else 0.0

        # 시설물별 기여 감소량 (단순 독립 분배: 각 시설의 단독 감소량)
        for fac in missing:
            r = REDUCTION_RATES.get(fac, 0)
            fac_saved = epdo_before * r
            facility_epdo_saved[fac] += fac_saved

        results.append({
            "grid_gid":            gid,
            "epdo_before":         epdo_before,
            "accident_cnt":        acc_cnt,
            "사망_cnt":            death_cnt,
            "중상_cnt":            injury_cnt,
            "gap_cnt":             gap_cnt,
            "missing_facilities":  " | ".join(sorted(missing)),
            "total_reduction_pct": round(total_reduction * 100, 2),
            "epdo_after":          epdo_after,
            "epdo_saved":          epdo_saved,
            "efficiency":          efficiency,
            "entropy_rank":        entropy_rank_map.get(gid, ""),
            "entropy_risk":        entropy_risk_map.get(gid, ""),
        })

print(f"    계산 완료: {len(results):,}개 격자")

## ── 3. 설치 우선순위: 효율성(시설물 1개당 EPDO 감소) 기준

In [ ]:
print("\n[3] 설치 우선순위 산출 중...")
results.sort(key=lambda x: x["efficiency"], reverse=True)
for i, row in enumerate(results, 1):
    row["install_priority"] = i

## ── 4. CSV 저장

In [ ]:
print("\n[4] CSV 저장 중...")
fieldnames = [
    "install_priority", "grid_gid", "epdo_before", "accident_cnt",
    "사망_cnt", "중상_cnt", "gap_cnt", "missing_facilities",
    "total_reduction_pct", "epdo_after", "epdo_saved", "efficiency",
    "entropy_rank", "entropy_risk",
]
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)
print(f"    저장 완료: {OUTPUT_PATH}")

## ── 5. 집계 요약 출력

In [ ]:
print("\n" + "="*60)
print("  STEP 11 결과 요약")
print("="*60)

total_epdo_before = sum(r["epdo_before"] for r in results)
total_epdo_saved  = sum(r["epdo_saved"]  for r in results)
total_grids       = len(results)
avg_reduction     = total_epdo_saved / total_epdo_before * 100 if total_epdo_before > 0 else 0

print(f"\n  분석 대상 격자 수 : {total_grids:,}개")
print(f"  현재 총 EPDO     : {total_epdo_before:,.0f}점")
print(f"  설치 후 예상 EPDO: {(total_epdo_before - total_epdo_saved):,.0f}점")
print(f"  총 예상 감소량   : {total_epdo_saved:,.0f}점 ({avg_reduction:.1f}%)")

print("\n  [시설물 유형별 총 기여 감소량]")
fac_sorted = sorted(facility_epdo_saved.items(), key=lambda x: x[1], reverse=True)
for fac, saved in fac_sorted:
    pct = saved / total_epdo_before * 100 if total_epdo_before > 0 else 0
    print(f"    {FACILITY_KR.get(fac, fac):12s}: {saved:8,.0f}점 ({pct:.1f}%)")

print("\n  [효율성 기준 설치 우선순위 Top-10]")
print(f"  {'순위':>4} {'격자ID':12} {'현재EPDO':>8} {'감소율':>6} {'EPDO감소':>8} {'효율성':>8} {'entropy순위':>8}")
for r in results[:10]:
    print(f"  {r['install_priority']:>4} {r['grid_gid']:12} "
          f"{r['epdo_before']:>8.0f} {r['total_reduction_pct']:>5.1f}% "
          f"{r['epdo_saved']:>8.0f} {r['efficiency']:>8.2f} "
          f"{r['entropy_rank']:>8}")

print("\n  [EPDO 절대 감소량 기준 Top-10]")
epdo_sorted = sorted(results, key=lambda x: x["epdo_saved"], reverse=True)
print(f"  {'순위':>4} {'격자ID':12} {'현재EPDO':>8} {'감소율':>6} {'EPDO감소':>8} {'entropy순위':>8}")
for i, r in enumerate(epdo_sorted[:10], 1):
    print(f"  {i:>4} {r['grid_gid']:12} "
          f"{r['epdo_before']:>8.0f} {r['total_reduction_pct']:>5.1f}% "
          f"{r['epdo_saved']:>8.0f} {r['entropy_rank']:>8}")

print("="*60)